### Authenticate
Get the client id and secret from env vars for prod.
In dev we can create a new client each time

In [1]:
import { registerSystem } from './helpers/gqlHandlers.ts'
import { authenticate, getTokenForSystemClient } from './helpers/authentication.ts'
import { CLIENT_ID, CLIENT_SECRET } from './helpers/vars.ts'
import { ADMIN_USERNAME, ADMIN_PASSWORD } from './helpers/vars.ts'


let clientId: string | null = null
let clientSecret: string | null = null

if (CLIENT_ID && CLIENT_SECRET) {
  clientId = CLIENT_ID
  clientSecret = CLIENT_SECRET
} else {
  // For dev mode
  const adminToken = await authenticate(ADMIN_USERNAME, ADMIN_PASSWORD)
  const systemRegistration = await registerSystem(adminToken)

  clientId = systemRegistration.data.registerSystem.system.clientId
  clientSecret = systemRegistration.data.registerSystem.clientSecret
}

const sysToken = await getTokenForSystemClient(clientId, clientSecret)

"dfedbc50-5c66-4b3c-a3fe-52b1df619af2"

### Sync locations

On update to v1.9, the postgres db won't have locations so this is required to sync them from mongo

In [2]:
import { syncLocations } from './helpers/gqlHandlers.ts'

const res = await syncLocations(sysToken)
res

"OK"

### Fetch birth and death events

In [3]:
import { fetchEvents } from './helpers/formsHandlers.ts'
import { extractFieldType } from './helpers/utils.ts'

const events = await fetchEvents(sysToken)

const ignoredFields = ['DIVIDER', 'PARAGRAPH']

const birthEvent = events.find((x) => x.id === 'birth')
const deathEvent = events.find((x) => x.id === 'death')

const birthEventFields = new Set(
  extractFieldType(birthEvent, 'fields')
    .filter((x) => !ignoredFields.includes(x.type))
    .map((f) => f.id)
    .filter((x) => x)
)

const deathEventFields = new Set(
  extractFieldType(deathEvent, 'fields')
    .filter((x) => !ignoredFields.includes(x.type))
    .map((f) => f.id)
    .filter((x) => x)
)


### Get all potential resolvers
Use only resolvers for used event fields to avoid nulls

In [ ]:
import defaultResolvers, {
  defaultBirthResolver,
  defaultDeathResolver,
} from './helpers/defaultResolvers.ts'
import { countryResolver, deathCountryResolver } from './countryData/countryResolvers.ts'

const birthAllResolvers = { ...defaultResolvers, ...countryResolver }
const deathAllResolvers = { ...defaultResolvers, ...countryResolver, ...deathCountryResolver }

const birthResolver = Object.fromEntries(
  Object.entries({...defaultBirthResolver, ...birthAllResolvers}).filter(([key, _value]) =>
    [...birthEventFields].includes(key)
  )
)

const deathResolver = Object.fromEntries(
  Object.entries({...defaultDeathResolver, ...deathAllResolvers}).filter(([key, _value]) =>
    [...deathEventFields].includes(key)
  )
)

### Migrate births

In [5]:
import { EVENT } from './helpers/vars.ts'
import {
  fetchBirthRegistration,
  bulkImport,
  fetchAllBirthRegistrations,
} from './helpers/gqlHandlers.ts'
import { transform } from './helpers/transform.ts'
import {
  batch,
  bulkImportIsolatingFailures,
  migrationProgress,
  getPaginationSkip,
  recordIndexErrors,
  formatErrorMessage,
} from './helpers/utils.ts'
import { RECORD_SKIP } from './helpers/vars.ts'

const migrateBirth = async (entryIds) => {
  const items = []

  for (const entryId of entryIds) {
    try {
      const birthRegistrationData = await fetchBirthRegistration(
        entryId,
        sysToken
      )
      if (!birthRegistrationData.data.fetchBirthRegistration) {
        console.error(JSON.stringify(birthRegistrationData, null, 2))
        migrationProgress.recordFailure(
          entryId,
          undefined,
          'No data from fetchBirthRegistration'
        )
        continue
      }

      const registration = birthRegistrationData.data.fetchBirthRegistration
      const trackingId = registration.registration?.trackingId

      try {
        const document = transform(registration, birthResolver, 'birth')
        items.push({ entryId, document })
      } catch (err) {
        migrationProgress.recordFailure(
          entryId,
          trackingId,
          `Transform error: ${formatErrorMessage(err)}`
        )
        continue
      }
    } catch (err) {
      migrationProgress.recordFailure(
        entryId,
        undefined,
        `Fetch error: ${formatErrorMessage(err)}`
      )
      continue
    }
  }

  if (items.length === 0) {
    return undefined
  }

  const indexResult = await bulkImportIsolatingFailures(
    items,
    sysToken,
    bulkImport
  )
  recordIndexErrors(indexResult, items)
  return indexResult
}

if (EVENT === 'birth') {
  const pageSize = 1000
  const batchSize = 100
  const { startPage, skipWithinPage } = getPaginationSkip(RECORD_SKIP, pageSize)
  let itemsRemaining = 0
  let page = startPage
  let totalProcessed = RECORD_SKIP

  migrationProgress.reset(RECORD_SKIP)

  if (RECORD_SKIP) {
    console.log(`Skipping first ${RECORD_SKIP} records`)
  }

  do {
    const birthRegistrations = await fetchAllBirthRegistrations(
      sysToken,
      page,
      pageSize
    )
    if (!birthRegistrations.data.searchEvents) {
      console.error(JSON.stringify(birthRegistrations, null, 2))
      throw new Error('No data from searchEvents')
    }

    const { results, totalItems } = birthRegistrations.data.searchEvents
    let birthIds = results.map((x) => x.id)

    if (page === startPage && skipWithinPage > 0) {
      console.log(
        `Skipping ${skipWithinPage} records at start of page ${page}`
      )
      birthIds = birthIds.slice(skipWithinPage)
    }

    console.log(
      `Processing next page of ${birthIds.length} of ${totalItems} total records`
    )

    const batches = batch(birthIds, batchSize)

    for (const recordBatch of batches) {
      await migrateBirth(recordBatch)
      console.log(
        `  - [${new Date().toISOString()}] Processed batch of ${recordBatch.length} records (total imported: ${migrationProgress.importedCount}, failed: ${migrationProgress.failedRecords.length})`
      )
    }

    totalProcessed += birthIds.length
    itemsRemaining = Math.max(0, totalItems - totalProcessed)

    console.log(
      `Processed ${totalProcessed} of ${totalItems} birth registrations... with ${itemsRemaining} remaining.`
    )
    page += 1
  } while (itemsRemaining > 0)

  migrationProgress.logSummary('birth')
}


Processing next page of 4 of 4 total records
  - [2026-06-15T11:23:17.551Z] Processed batch of 4 records (total imported: 4, failed: 0)
Processed 4 of 4 birth registrations... with 0 remaining.

MIGRATION COMPLETE: birth
Imported: 4
Failed: 0


### Migrate Deaths

In [6]:
import {
  fetchDeathRegistration,
  fetchAllDeathRegistrations,
} from './helpers/gqlHandlers.ts'

const migrateDeath = async (entryIds) => {
  const items = []

  for (const entryId of entryIds) {
    try {
      const deathRegistrationData = await fetchDeathRegistration(
        entryId,
        sysToken
      )
      if (!deathRegistrationData.data.fetchDeathRegistration) {
        console.error(JSON.stringify(deathRegistrationData, null, 2))
        migrationProgress.recordFailure(
          entryId,
          undefined,
          'No data from fetchDeathRegistration'
        )
        continue
      }

      const registration = deathRegistrationData.data.fetchDeathRegistration
      const trackingId = registration.registration?.trackingId

      try {
        const document = transform(registration, deathResolver, 'death')
        items.push({ entryId, document })
      } catch (err) {
        migrationProgress.recordFailure(
          entryId,
          trackingId,
          `Transform error: ${formatErrorMessage(err)}`
        )
        continue
      }
    } catch (err) {
      migrationProgress.recordFailure(
        entryId,
        undefined,
        `Fetch error: ${formatErrorMessage(err)}`
      )
      continue
    }
  }

  if (items.length === 0) {
    return undefined
  }

  const indexResult = await bulkImportIsolatingFailures(
    items,
    sysToken,
    bulkImport
  )
  recordIndexErrors(indexResult, items)
  return indexResult
}

if (EVENT === 'death') {
  const pageSize = 1000
  const batchSize = 100
  const { startPage, skipWithinPage } = getPaginationSkip(RECORD_SKIP, pageSize)
  let itemsRemaining = 0
  let page = startPage
  let totalProcessed = RECORD_SKIP

  migrationProgress.reset(RECORD_SKIP)

  if (RECORD_SKIP) {
    console.log(`Skipping first ${RECORD_SKIP} records`)
  }

  do {
    const deathRegistrations = await fetchAllDeathRegistrations(
      sysToken,
      page,
      pageSize
    )
    if (!deathRegistrations.data.searchEvents) {
      console.error(JSON.stringify(deathRegistrations, null, 2))
      throw new Error('No data from searchEvents')
    }
    const { results, totalItems } = deathRegistrations.data.searchEvents
    let deathIds = results.map((x) => x.id)

    if (page === startPage && skipWithinPage > 0) {
      console.log(
        `Skipping ${skipWithinPage} records at start of page ${page}`
      )
      deathIds = deathIds.slice(skipWithinPage)
    }

    console.log(
      `Processing next page of ${deathIds.length} of ${totalItems} total records`
    )

    const batches = batch(deathIds, batchSize)

    for (const recordBatch of batches) {
      await migrateDeath(recordBatch)
      console.log(
        `  - [${new Date().toISOString()}] Processed batch of ${recordBatch.length} records (total imported: ${migrationProgress.importedCount}, failed: ${migrationProgress.failedRecords.length})`
      )
    }

    totalProcessed += deathIds.length
    itemsRemaining = Math.max(0, totalItems - totalProcessed)

    console.log(
      `Processed ${totalProcessed} of ${totalItems} death registrations... with ${itemsRemaining} remaining.`
    )
    page += 1
  } while (itemsRemaining > 0)

  migrationProgress.logSummary('death')
}


In [7]:
import { reindex } from "./helpers/gqlHandlers.ts";

const reindexResponse = await reindex(sysToken);
reindexResponse


"OK"

### Output results


In [8]:
import { getMigrationSummaryPath } from './helpers/vars.ts'

if (migrationProgress.failedRecords.length > 0) {
  console.log(
    `Migration finished with ${migrationProgress.failedRecords.length} failed record(s). See failed records summary above.`
  )
} else {
  console.log('Declarations successfully migrated with no failures.')
}

console.log(`Downloadable summary: ${getMigrationSummaryPath()}`)


Declarations successfully migrated with no failures.
